# 03 — Entropy Geodesics (CI‑Aware)
**Abstract.**  
Integrate entropy geodesics on the symbolic manifold using `src.entropy.entropy_geodesics` (or a safe fallback).  
This CI‑aware version loads projection points from:
- results/projection_points.csv (preferred)
- CI‑generated Cremona labels (synthesizing projection points)
- Synthetic fallback (local development)

It then computes geodesic trajectories, visualizes them, and computes a simple divergence stability metric.


In [ ]:
import sys, os
sys.path.append('src')
sys.path.append('../src')
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

np.random.seed(123)
os.makedirs('results', exist_ok=True)


In [ ]:
# --- CI + arithmetic archive detection ---
RUNNING_IN_CI = os.environ.get("GITHUB_ACTIONS", "false") == "true"

CI_LABELS = "data/raw/ci_subset.csv"
EC_DATA = "external/ecdata"
LMFDB = "external/lmfdb"

print("Running in CI:", RUNNING_IN_CI)
print("CI labels:", os.path.exists(CI_LABELS))
print("Cremona ecdata:", os.path.exists(EC_DATA))
print("LMFDB:", os.path.exists(LMFDB))


In [ ]:
if os.path.exists('results/projection_points.csv'):
    df = pd.read_csv('results/projection_points.csv')
    print("Loaded results/projection_points.csv")

elif RUNNING_IN_CI and os.path.exists(CI_LABELS):
    print("CI mode: synthesizing projection points from CI labels")
    labels = pd.read_csv(CI_LABELS)
    n = len(labels)
    df = pd.DataFrame({
        'x': np.random.rand(n),
        'y': np.random.rand(n),
        'z': np.random.rand(n)
    })

else:
    print("Projection points not found; generating synthetic points.")
    n = 150
    df = pd.DataFrame({
        'x': np.random.randint(0,4,size=n),
        'y': np.random.lognormal(5,1.0,size=n),
        'z': np.random.exponential(1.0,size=n)
    })

points = df[['x','y','z']].values


In [ ]:
try:
    from src.entropy.entropy_geodesics import EntropyGeodesics
    G = EntropyGeodesics()

    trajectories = []
    for i in range(5):
        idx = np.random.randint(0, len(points))
        x0 = points[idx]
        v0 = np.random.normal(scale=0.1, size=3)
        traj = G.geodesic(x0, v0, steps=80)
        trajectories.append(traj)

    trajectories = np.array(trajectories)
    np.save('results/geodesics.npy', trajectories)

    print("Computed geodesics using src.entropy.entropy_geodesics")

except Exception as e:
    print("EntropyGeodesics not available; using simple straight-line fallback.", e)

    trajectories = []
    for i in range(5):
        idx = np.random.randint(0, len(points))
        x0 = points[idx]
        v0 = np.random.normal(scale=0.05, size=3)
        traj = np.array([x0 + t*v0 for t in np.linspace(0,1,80)])
        trajectories.append(traj)

    trajectories = np.array(trajectories)
    np.save('results/geodesics.npy', trajectories)


In [ ]:
fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(points[:,0], points[:,1], points[:,2],
           c='lightgray', s=8, alpha=0.6)

colors = ['C0','C1','C2','C3','C4']
for i, traj in enumerate(trajectories):
    ax.plot(traj[:,0], traj[:,1], traj[:,2],
            color=colors[i], lw=2, label=f'geodesic_{i}')

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('z')
plt.title('Entropy Geodesics on Symbolic Manifold')
plt.legend()

plt.savefig('results/geodesics_overlay.png', dpi=200)
plt.show()

print("Saved results/geodesics_overlay.png and results/geodesics.npy")


In [ ]:
# compute average pairwise distance between geodesic endpoints
endpoints = np.array([traj[-1] for traj in trajectories])

dists = np.linalg.norm(endpoints[:,None,:] - endpoints[None,:,:], axis=2)
avg_divergence = np.mean(dists[np.triu_indices(len(dists), k=1)])

print("Average geodesic endpoint divergence:", avg_divergence)


**Interpretation.**  
Geodesic trajectories show how symbolic flows evolve under the entropy metric.  
Divergence quantifies sensitivity to initial conditions.  
Next: compute metric perturbations (Notebook 04).


In [ ]:
print("Notebook: 03_entropy_geodesics")
print("Data source:", 
      "projection_points.csv" if os.path.exists('results/projection_points.csv')
      else "CI labels" if RUNNING_IN_CI else "synthetic")
print("Outputs: results/geodesics.npy, results/geodesics_overlay.png")
